# Real video -> real 3D scene

Runs the actual full pipeline on a real phone video: frame extraction + blur
filtering (same code as `backend/app/pipeline.py`), COLMAP SfM, and Gaussian
Splatting training (same code as `worker/runner.py`) -- the same building
blocks already proven on the synthetic scene, now on real, unknown footage.

**Runtime > Change runtime type > GPU (T4 is fine)** before running these cells.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU attached -- go to Runtime > Change runtime type > GPU, then re-run.')

In [ ]:
!git clone https://github.com/yusupildan-wq/Scene-Reconstruction.git
%cd Scene-Reconstruction

In [ ]:
# torch is already installed (with CUDA) by Colab -- do not reinstall it here.
!pip install -q pycolmap gsplat scipy opencv-python-headless

## Upload your video

Run the next cell, then use the "Choose Files" button that appears to upload
your video from your phone/computer.

In [ ]:
from google.colab import files
uploaded = files.upload()
video_filename = next(iter(uploaded.keys()))
print('Uploaded:', video_filename)

## Frame extraction (same code as the real backend)

Samples every 10th frame and drops blurry ones -- identical logic to
`backend/app/pipeline.py`, imported directly rather than duplicated.

In [ ]:
import sys
sys.path.insert(0, 'backend')
sys.path.insert(0, 'worker')

from pathlib import Path
from app.pipeline import extract_frames

frames_dir = Path('real_video_frames')
result = extract_frames(Path(video_filename), frames_dir)
print(f'{result.total_frames_seen} frames seen, {result.selected_frame_count} selected after blur/redundancy filtering')

## Run real COLMAP SfM on the real frames

Unlike the synthetic scene, this has no ground truth to check against --
it either finds a good reconstruction or it doesn't. A real test of the
pipeline's real-world robustness, not just its mechanics.

In [ ]:
import pycolmap

database_path = Path('real_video_colmap.db')
sparse_dir = Path('real_video_sparse')
sparse_dir.mkdir(exist_ok=True)

pycolmap.extract_features(str(database_path), str(frames_dir))
pycolmap.match_exhaustive(str(database_path))
reconstructions = pycolmap.incremental_mapping(str(database_path), str(frames_dir), str(sparse_dir))

if not reconstructions:
    raise RuntimeError('COLMAP could not register any cameras -- see notes below on why this can happen.')

best_id = max(reconstructions, key=lambda k: reconstructions[k].num_reg_images())
recon = reconstructions[best_id]
print(f'Registered {recon.num_reg_images()} / {result.selected_frame_count} frames')
print(f'Triangulated {recon.num_points3D()} 3D points')
print(f'Mean reprojection error: {recon.compute_mean_reprojection_error():.3f} px')

If that cell raises "could not register any cameras," common real-world
causes: not enough visual overlap between frames (moved/turned too fast),
textureless surfaces (blank walls), motion blur, or too few selected frames.
This is a legitimate, expected failure mode to hit and learn from, not
necessarily a bug -- try a slower, more deliberate recording with more
surface texture in view if it happens.

In [ ]:
from runner import SfmResult, train_gaussian_splatting

sfm = SfmResult(reconstruction=recon, images_dir=frames_dir)
# Real room-scale detail needs real training budget: 20,000 iterations instead
# of 3,000-6,000. densify_until stops adding new Gaussians for the last 4,000
# iterations (standard practice) so training ends with refinement only, not
# still-unrefined brand-new Gaussians. Expect this to take considerably
# longer than previous runs -- meaningfully more real time, not a quick step.
gaussians = train_gaussian_splatting(sfm, num_iterations=20000, densify_until=16000)
print('Trained', gaussians['means'].shape[0], 'Gaussians')

In [ ]:
import json

export = {
    'means': gaussians['means'].tolist(),
    'quats': gaussians['quats'].tolist(),
    'scales': gaussians['scales'].tolist(),
    'opacities': gaussians['opacities'].tolist(),
    'colors': gaussians['colors'].tolist(),
}
with open('real_scene.json', 'w') as f:
    json.dump(export, f)

files.download('real_scene.json')